# LD-BD Separation LP Visualization (Interactive 3D)

This notebook logs separation LP results per iteration and renders interactive 3D plots.
- Planes: all cut planes generated in the current iteration.
- Points: all anchors (blue) and neighbors (light blue) known up to the current iteration.
- Master point: purple.

Each iteration also saves a PNG image.

In [1]:
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyomo.environ as pyo
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from pyomo.contrib.gdpopt.discrete_search_enums import SearchPhase
from pyomo.contrib.gdpopt.ldbd import GDP_LDBD_Solver

from toy_problem import build_model


get_ipython().run_line_magic("matplotlib", "qt")


In [5]:
@dataclass
class SeparationRecord:
    iteration: int
    anchor: tuple
    sep_obj: float
    alpha: float
    p_vals: tuple


@dataclass
class MasterRecord:
    iteration: int
    point: tuple
    z_lb: float | None


class LDBDSeparationLogger(GDP_LDBD_Solver):
    def __init__(self):
        super().__init__()
        self.separation_records = []
        self.master_records = []

    def _solve_separation_lp(self, anchor_point, config):
        p_vals, alpha_val = super()._solve_separation_lp(anchor_point, config)
        if p_vals is None:
            return p_vals, alpha_val

        anchor_point = tuple(anchor_point)
        sep_obj = sum(p_vals[i] * anchor_point[i] for i in range(len(p_vals))) + alpha_val
        self.separation_records.append(
            SeparationRecord(
                iteration=int(getattr(self, "iteration", 0)),
                anchor=anchor_point,
                sep_obj=float(sep_obj),
                alpha=float(alpha_val),
                p_vals=tuple(float(v) for v in p_vals),
            )
        )
        return p_vals, alpha_val

    def _solve_master(self, config):
        z_lb, next_point = super()._solve_master(config)
        if next_point is not None:
            self.master_records.append(
                MasterRecord(
                    iteration=int(getattr(self, "iteration", 0)),
                    point=tuple(next_point),
                    z_lb=float(z_lb) if z_lb is not None else None,
                )
            )
        return z_lb, next_point

In [6]:
def plot_iteration_planes(records, master_records, point_info, external_bounds, output_dir, ite=None, uplimit=None):
    if not records or not master_records:
        raise RuntimeError("Missing separation LP or master records for plotting.")

    if len(external_bounds) < 2:
        raise RuntimeError("Expected at least two external variables for 3D plotting.")

    lb1, ub1 = external_bounds[0]
    lb2, ub2 = external_bounds[1]
    grid_x = np.linspace(lb1, ub1, 21)
    grid_y = np.linspace(lb2, ub2, 21)
    mesh_x, mesh_y = np.meshgrid(grid_x, grid_y)

    records_by_iter = {}
    for record in records:
        records_by_iter.setdefault(record.iteration, []).append(record)
    master_by_iter = {m.iteration: m for m in master_records}

    max_iter = max(records_by_iter.keys() & master_by_iter.keys())
    if ite is not None:
        max_iter = min(max_iter, int(ite))

    for iteration in range(1, max_iter + 1):
        iter_records = records_by_iter.get(iteration, [])
        master = master_by_iter.get(iteration)
        if master is None:
            continue

        fig = plt.figure(figsize=(7, 5))
        ax = fig.add_subplot(111, projection="3d")

        for idx, rec in enumerate(iter_records):
            p0 = rec.p_vals[0]
            p1 = rec.p_vals[1]
            alpha = rec.alpha
            plane_z = p0 * mesh_x + p1 * mesh_y + alpha
            ax.plot_surface(
                mesh_x,
                mesh_y,
                plane_z,
                color="salmon",
                alpha=0.5,
                linewidth=0,
                antialiased=True,
                label="Cut planes" if idx == 0 else None,
            )

        anchor_x = []
        anchor_y = []
        anchor_z = []
        neighbor_x = []
        neighbor_y = []
        neighbor_z = []
        for point, info in (point_info or {}).items():
            if info.get("iteration_found") is None:
                continue
            if info.get("iteration_found") > iteration:
                continue
            source = info.get("source")
            if source == str(SearchPhase.ANCHOR):
                anchor_x.append(point[0])
                anchor_y.append(point[1])
                anchor_z.append(float(info.get("objective")))
            elif source == str(SearchPhase.NEIGHBOR_EVAL):
                neighbor_x.append(point[0])
                neighbor_y.append(point[1])
                neighbor_z.append(float(info.get("objective")))

        if anchor_x:
            ax.scatter(
                anchor_x,
                anchor_y,
                anchor_z,
                color="blue",
                s=45,
                label="Anchors",
            )
        if neighbor_x:
            ax.scatter(
                neighbor_x,
                neighbor_y,
                neighbor_z,
                color="#7ec8ff",
                s=35,
                label="Neighbors",
            )

        point_x = master.point[0]
        point_y = master.point[1]
        if master.z_lb is not None:
            point_z = master.z_lb
        elif iter_records:
            p0 = iter_records[0].p_vals[0]
            p1 = iter_records[0].p_vals[1]
            alpha = iter_records[0].alpha
            point_z = p0 * point_x + p1 * point_y + alpha
        else:
            point_z = 0.0

        ax.scatter(
            [point_x],
            [point_y],
            [point_z],
            color="purple",
            s=55,
            label="Master point",
        )

        ax.set_xlabel("e1")
        ax.set_ylabel("e2")
        ax.set_zlabel("cut plane")
        ax.set_title(f"Iteration {iteration}")
        ax.legend(loc="upper left")
        fig.tight_layout()
        if uplimit is not None:
            zmin, _ = ax.get_zlim()
            ax.set_zlim(zmin, float(uplimit))

        output_path = output_dir / f"separation_plane_iter_{iteration:02d}.png"
        fig.savefig(output_path, dpi=300)
        plt.show()

In [12]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from pyomo.contrib.gdpopt.discrete_search_enums import SearchPhase

def plot_iteration_planes(records, master_records, point_info, external_bounds, output_dir, ite=None, uplimit=None):
    if not records or not master_records:
        raise RuntimeError("Missing separation LP or master records for plotting.")

    if len(external_bounds) < 2:
        raise RuntimeError("Expected at least two external variables for 3D plotting.")

    lb1, ub1 = external_bounds[0]
    lb2, ub2 = external_bounds[1]
    
    grid_x = np.linspace(lb1, ub1, 21)
    grid_y = np.linspace(lb2, ub2, 21)
    mesh_x, mesh_y = np.meshgrid(grid_x, grid_y)

    records_by_iter = {}
    for record in records:
        records_by_iter.setdefault(record.iteration, []).append(record)
    master_by_iter = {m.iteration: m for m in master_records}

    max_iter = max(records_by_iter.keys() & master_by_iter.keys())
    if ite is not None:
        max_iter = min(max_iter, int(ite))

    for iteration in range(1, max_iter + 1):
        iter_records = records_by_iter.get(iteration, [])
        master = master_by_iter.get(iteration)
        if master is None:
            continue

        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111, projection="3d")

        # --- 1. Plot Cut Planes ---
        for idx, rec in enumerate(iter_records):
            p0 = rec.p_vals[0]
            p1 = rec.p_vals[1]
            alpha = rec.alpha
            plane_z = p0 * mesh_x + p1 * mesh_y + alpha
            
            # Surface 渲染
            surf = ax.plot_surface(
                mesh_x,
                mesh_y,
                plane_z,
                color="pink",      
                alpha=0.3,         
                linewidth=0,
                antialiased=True,
            )
            # 为了图例兼容性，隐式绘制一小段不可见的线来代表 Cut Plane
            if idx == 0:
                ax.plot([], [], [], color="pink", alpha=0.6, linewidth=5, label="Cut planes")

        anchor_x, anchor_y, anchor_z = [], [], []
        neighbor_x, neighbor_y, neighbor_z = [], [], []
        
        for point, info in (point_info or {}).items():
            if info.get("iteration_found") is None:
                continue
            if info.get("iteration_found") > iteration:
                continue
                
            source = info.get("source")
            if source == str(SearchPhase.ANCHOR):
                anchor_x.append(point[0])
                anchor_y.append(point[1])
                anchor_z.append(float(info.get("objective")))
            elif source == str(SearchPhase.NEIGHBOR_EVAL):
                neighbor_x.append(point[0])
                neighbor_y.append(point[1])
                neighbor_z.append(float(info.get("objective")))

        # --- 2. Plot Anchors (Dark Green) and Neighbors ---
        if anchor_x:
            ax.scatter(
                anchor_x, anchor_y, anchor_z,
                color="darkgreen",  # 这一轮改为深绿色
                edgecolor="white",  # 白色边缘提升立体感和对比度
                s=90,               
                alpha=1.0,
                depthshade=False,
                label="Anchors"     # 添加图例标签
            )
        if neighbor_x:
            ax.scatter(
                neighbor_x, neighbor_y, neighbor_z,
                color="lime",       # 邻居保持亮绿色
                edgecolor="green",
                s=70,
                alpha=0.8,
                depthshade=False,
                label="Neighbors"   # 添加图例标签
            )

        # --- 3. Plot Master Point ---
        point_x = master.point[0]
        point_y = master.point[1]
        if master.z_lb is not None:
            point_z = master.z_lb
        elif iter_records:
            p0 = iter_records[0].p_vals[0]
            p1 = iter_records[0].p_vals[1]
            alpha = iter_records[0].alpha
            point_z = p0 * point_x + p1 * point_y + alpha
        else:
            point_z = 0.0

        ax.scatter(
            [point_x], [point_y], [point_z],
            color="magenta",
            marker="s",         
            s=80,               
            alpha=1.0,
            depthshade=False,
            label="Master point" # 添加图例标签
        )

        # --- 4. Style Adjustments & Legend ---
        ax.set_xlabel("$e_1$", fontsize=14, labelpad=10)
        ax.set_ylabel("$e_2$", fontsize=14, labelpad=10)
        ax.set_zlabel("Objective function", fontsize=14, labelpad=10)
        ax.set_title(f"Iteration $k = {iteration}$", fontsize=16)

        ax.set_xlim(0.5, 5.5)
        ax.set_ylim(0.5, 5.5)
        ax.set_zlim(-2.1, 0)
        
        ax.set_xticks([1, 2, 3, 4, 5])
        ax.set_yticks([1, 2, 3, 4, 5])

        ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
        ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
        ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))

        ax.view_init(elev=20, azim=45)

        # 【新增】：在此处添加图例，并设置位置和背景透明度
        ax.legend(loc="upper left", fontsize=11, framealpha=0.8, edgecolor="gray")

        fig.tight_layout()

        # Save and Show
        output_path = output_dir / f"separation_plane_iter_{iteration:02d}.png"
        fig.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.show()

In [13]:
output_dir = Path(".").resolve()
ite = None  # set an int to cap iterations, e.g., 10
uplimit = 1  # set a float to cap the z-axis upper limit
model = build_model()
solver = LDBDSeparationLogger()

results = solver.solve(
    model,
    starting_point=[5, 1],
    direction_norm="Linf",
    logical_constraint_list=[model.oneY1, model.oneY2],
    subproblem_solver="gams",
    mip_solver="gurobi",
    separation_solver="gurobi",
    tee=True,
 )

external_info = getattr(solver.data_manager, "external_var_info_list", []) or []
external_bounds = [(info.LB, info.UB) for info in external_info]
plot_iteration_planes(
    solver.separation_records,
    solver.master_records,
    getattr(solver.data_manager, "point_info", None),
    external_bounds,
    output_dir,
    ite=ite,
    uplimit=uplimit,
 )

print("Solver Status:", results.solver.status)
print("Solver Termination Condition:", results.solver.termination_condition)
print("Images saved to:", output_dir)

Starting GDPopt version 22.5.13 using LDBD algorithm
iterlim: None
time_limit: None
tee: true
logger: <Logger pyomo.contrib.gdpopt (INFO)>
mip_solver: gurobi
mip_solver_args:
subproblem_solver: gams
subproblem_solver_args:
direction_norm: Linf
starting_point: [5, 1]
logical_constraint_list: ComponentSet(['oneY1 (key=oneY1)', 'oneY2 (key=oneY2)'])
disjunction_list: None
infinity_output: 100000000.0
integer_tolerance: 1e-05
constraint_tolerance: 1e-06
variable_tolerance: 1e-08
subproblem_initialization_method: <function restore_vars_to_original_values at 0x0000025B156BC9A0>
call_before_subproblem_solve: <class 'type'>
call_after_subproblem_solve: <class 'type'>
call_after_subproblem_feasible: <class 'type'>
force_subproblem_nlp: false
subproblem_presolve: true
tighten_nlp_var_bounds: false
round_discrete_vars: true
max_fbbt_iterations: 3
bound_tolerance: 1e-06
separation_solver: gurobi
separation_solver_args:
cuts_for_all_evaluated_points: false

If you use this software, you may cite th